In [0]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

num_events = 50000

df = (
    spark.range(num_events)
    .withColumn(
        "vessel_id",
        F.concat(F.lit("V"), F.lpad((F.rand() * 500).cast("int"), 4, "0"))
    )
    .withColumn(
        "event_timestamp",
        F.expr("timestamp('2025-01-01 00:00:00')") + F.expr("INTERVAL 1 MINUTE") * (F.rand() * 43200).cast("int")
    )
    .withColumn(
    "latitude",
    F.when(F.rand() < 0.01, F.lit(999.0))
     .otherwise(F.round(F.lit(37.5) + (F.rand() * 1.0), 6))
    )
    .withColumn(
    "longitude",
    F.when(F.rand() < 0.01, F.lit(-999.0))
     .otherwise(F.round(F.lit(23.0) + (F.rand() * 1.5), 6))
    )
    .withColumn(
        "speed_knots",
        F.round(F.rand() * 25, 2)
    )
    .withColumn(
        "heading",
        F.round(F.rand() * 360, 2)
    )
    .drop("id")
)


########################################################################################3
#landing not working cause of Workspace Security Polizy Error. My workspace has disabled public DBFS root access (dbfs:/), which means you can't write directly to paths like dbfs:/tmp/
#####################
# landing_path = "dbfs:/tmp/maritime_lakehouse/sample_ais_events"
# df.write.mode("overwrite").parquet(landing_path)
#####################
########################################################################################

bronze_df = (
    df
    .withColumn("source_system", F.lit("synthetic_ais"))
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("record_id", F.expr("uuid()"))
)

display(bronze_df.limit(20))

#save as delta table
bronze_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("bronze_ais_synthetic")